# makemore Bigram — Part 2 (neural net)

Follows Part 1. Same `names.txt`, start/end token `.` (index 0).


In [ ]:
words = open('names.txt', 'r').read().splitlines()


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Vocab + count matrix (for comparison with Part 1)
N = torch.zeros((27, 27), dtype=torch.int32)
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

P = (N + 1).float()
P /= P.sum(1, keepdim=True)


## Toy example: one name (`emma`)

Build `(xs, ys)` and walk through forward pass + loss on 5 bigrams.


In [ ]:
xs, ys = [], []
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.numel()
print(xs.shape, ys.shape)
xs, ys


In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)


In [ ]:
# Forward pass (softmax via exp — same as the lecture)
xenc = F.one_hot(xs, num_classes=27).float()
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)

plt.imshow(xenc.numpy())
plt.title('one-hot inputs (5 x 27)')
plt.axis('off');


In [ ]:
nlls = torch.zeros(num)
for i in range(num):
    x = xs[i].item()
    y = ys[i].item()
    print('--------')
    print(f'bigram {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
    p = probs[i, y]
    logp = torch.log(p)
    nll = -logp
    print(f'prob={p.item():.4f}  nll={nll.item():.4f}')
    nlls[i] = nll

print('=========')
print('average NLL (loss) =', nlls.mean().item())


## Single-step gradient descent


In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

xenc = F.one_hot(xs, num_classes=27).float()
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss = -probs[torch.arange(num), ys].log().mean()
loss


In [ ]:
W.grad = None
loss.backward()
W.data += -0.01 * W.grad
loss.item()


## Full dataset + training loop


In [ ]:
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)


In [ ]:
for k in range(100):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()

    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

    if k % 10 == 0 or k == 99:
        print(f'step {k:3d}: loss = {loss.item():.4f}')


## Compare to smoothed count model (Part 1)


In [ ]:
with torch.no_grad():
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    nn_nll = -probs[torch.arange(num), ys].log().mean()
    count_nll = -P[xs, ys].log().mean()

print(f'count model avg NLL: {count_nll.item():.4f}')
print(f'neural net avg NLL:  {nn_nll.item():.4f}')


## Sample names from trained weights


In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))
